In [24]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
plt.style.use('default')
plt.rcParams['font.sans-serif']=['SimHei']
plt.rcParams['axes.unicode_minus']=False 
import pickle
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Model
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.layers import Conv1D,Conv2D,MaxPooling1D, MaxPooling2D, Flatten, Dense, Input, Dropout, LSTM, TimeDistributed, Reshape, BatchNormalization, Activation, concatenate
cm = 1/2.54
# 字号
fontsize = 8
seed = 2025

In [25]:
def get_xy(Cdata, Hdata):
    input_1 = Cdata.iloc[:, :-3]
    input_2 = Hdata.iloc[:, :-3]
    output = Cdata.iloc[:, -3:]
    return input_1, input_2, output
# 归一化函数
def data_normlization(train_data, val_data, test_data):
    scaler = StandardScaler()
    train_data = scaler.fit_transform(train_data)
    val_data = scaler.transform(val_data)
    test_data = scaler.transform(test_data)
    return train_data, val_data, test_data

### 中期融合

#### 模型

In [26]:
def CNN_model(input_shape_1, input_shape_2):
    # 曲线输入
    input_1 = Input(shape=input_shape_1)
    x = Conv1D(filters=16, kernel_size=5, strides=5, activation='relu')(input_1)
    x = MaxPooling1D(pool_size=2, strides=2)(x)
    x = Conv1D(filters=32, kernel_size=5, strides=2, activation='relu')(x)
    x = Conv1D(filters=32, kernel_size=3, strides=1, activation='relu')(x)
    x = Conv1D(filters=64, kernel_size=3, strides=1, activation='relu')(x)
    x = MaxPooling1D(pool_size=2, strides=1)(x)
    x = Flatten()(x)
    output_1 = Dense(32, activation='relu')(x)
    # 特征输入
    input_2 = Input(shape=input_shape_2)
    output_2 = Dense(32, activation='relu')(input_2)
    # 合并
    x = concatenate([output_1, output_2])
    x = Dropout(0.1)(x)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.1)(x)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.1)(x)
    x = Dense(1, activation='relu')(x)
    model = Model(inputs=[input_1, input_2], outputs=x)
    model.compile(optimizer='adam', loss='mse')
    return model
def model_train(model, Ctrain_x, Htrain_x, train_y, Cval_x, Hval_x, val_y, epochs):
    train_y = train_y.iloc[:, -2]/3500 # 计算SOH
    val_y = val_y.iloc[:, -2]/3500
    Ctrain_x = Ctrain_x.reshape(Ctrain_x.shape[0], Ctrain_x.shape[1], 1) # 修改形状
    Cval_x = Cval_x.reshape(Cval_x.shape[0], Cval_x.shape[1], 1)
    model.fit([Ctrain_x, Htrain_x], train_y, epochs=epochs, validation_data=([Cval_x, Hval_x], val_y), verbose=0)
    return model
def model_eval(model, Ctest_x, Htest_x, test_y):
    Ctest_x = Ctest_x.reshape(Ctest_x.shape[0], Ctest_x.shape[1], 1)
    y_pred = model.predict([Ctest_x, Htest_x])
    # 修改形状
    y_pred = y_pred.reshape(y_pred.shape[0])
    y_pred = pd.DataFrame(y_pred, columns=['pred'])
    # 将test_y的capacity列转变为SOH
    test_y['SOH'] = test_y['capacity']/3500
    test_y.reset_index(drop=True, inplace=True)
    result = pd.concat([test_y, y_pred], axis=1)
    return result

#### CC阶段

In [27]:
# 加载数据集，Ctrain_CC表示曲线训练集的CC数据，Htrain_CC表示曲线训练集的HI数据
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\5-fusion\data_CC.pkl', 'rb') as file:
    Ctrain_CC, Cval_CC, Ctest_CC, Htrain_CC, Hval_CC, Htest_CC = pickle.load(file)
    file.close()
# 划分输入输出
train_x1, train_x2, train_y = get_xy(Ctrain_CC, Htrain_CC)
val_x1, val_x2, val_y = get_xy(Cval_CC, Hval_CC)
test_x1, test_x2, test_y = get_xy(Ctest_CC, Htest_CC)
# 归一化
train_x1, val_x1, test_x1 = data_normlization(train_x1, val_x1, test_x1)
train_x2, val_x2, test_x2 = data_normlization(train_x2, val_x2, test_x2)

In [28]:
train_x2.shape

(13762, 3)

In [29]:
model_CC = CNN_model(input_shape_1=(train_x1.shape[1], 1), input_shape_2=(train_x2.shape[1],))
# CC
epochs = 100
model_CC = model_train(model_CC, train_x1, train_x2, train_y, val_x1, val_x2, val_y, epochs)
# 保存模型
model_CC.save(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\5-fusion\rusult\middle\model_CC.h5')
result_CC = model_eval(model_CC, test_x1, test_x2, test_y)
result_CC.to_csv(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\5-fusion\rusult\middle\result_CC.csv', index=False)


127/127 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


#### CV阶段

In [30]:
# 加载数据集
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\5-fusion\data_CV.pkl', 'rb') as file:
    Ctrain_CV, Cval_CV, Ctest_CV, Htrain_CV, Hval_CV, Htest_CV = pickle.load(file)
    file.close()
# 划分输入输出
train_x1, train_x2, train_y = get_xy(Ctrain_CV, Htrain_CV)
val_x1, val_x2, val_y = get_xy(Cval_CV, Hval_CV)
test_x1, test_x2, test_y = get_xy(Ctest_CV, Htest_CV)
# 归一化
train_x1, val_x1, test_x1 = data_normlization(train_x1, val_x1, test_x1)
train_x2, val_x2, test_x2 = data_normlization(train_x2, val_x2, test_x2)

In [31]:
model_CV = CNN_model(input_shape_1=(train_x1.shape[1], 1), input_shape_2=(train_x2.shape[1],))
epochs = 100
model_CV = model_train(model_CV, train_x1, train_x2, train_y, val_x1, val_x2, val_y, epochs)
# 保存模型
model_CV.save(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\5-fusion\rusult\middle\model_CV.h5')
result_CV = model_eval(model_CV, test_x1, test_x2, test_y)
result_CV.to_csv(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\5-fusion\rusult\middle\result_CV.csv', index=False)


127/127 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


#### RE阶段

In [32]:
# 加载数据集
with open(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\5-fusion\data_RE.pkl', 'rb') as file:
    Ctrain_RE, Cval_RE, Ctest_RE, Htrain_RE, Hval_RE, Htest_RE = pickle.load(file)
    file.close()
# 划分输入输出
train_x1, train_x2, train_y = get_xy(Ctrain_RE, Htrain_RE)
val_x1, val_x2, val_y = get_xy(Cval_RE, Hval_RE)
test_x1, test_x2, test_y = get_xy(Ctest_RE, Htest_RE)
# 归一化
train_x1, val_x1, test_x1 = data_normlization(train_x1, val_x1, test_x1)
train_x2, val_x2, test_x2 = data_normlization(train_x2, val_x2, test_x2)

In [33]:
model_RE = CNN_model(input_shape_1=(train_x1.shape[1], 1), input_shape_2=(train_x2.shape[1],))
epochs = 100
model_RE = model_train(model_RE, train_x1, train_x2, train_y, val_x1, val_x2, val_y, epochs)
# 保存模型
model_RE.save(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\5-fusion\rusult\middle\model_RE.h5')
result_RE = model_eval(model_RE, test_x1, test_x2, test_y)
result_RE.to_csv(r'G:\boshi\1_boyi\3_ziji\My_program\long\3\code\5-fusion\rusult\middle\result_RE.csv', index=False)


127/127 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
